# EMBER-Dyn Quickstart 04 — Random Forest (template)

**Dataset**: EMBER-Dyn (T1 binary detection)  
**Expected metric**: test ROC AUC in the ~0.97--0.99 range; exact
value depends on tree count and depth.  
**Runtime**: ~3 minutes on Colab free CPU tier

Template for users who want a Random Forest baseline. The paper
Table 4 reports LightGBM only; use this notebook to produce a
Random Forest row for your own comparison.

In [ ]:
!pip install -q pandas pyarrow scikit-learn

In [ ]:
import pandas as pd, numpy as np
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# EMBER-Dyn open tier is archived on Zenodo (DOI in the README). Download it first
# (e.g. `pip install zenodo_get && zenodo_get <ZENODO_DOI>`) and point DATA_DIR at the
# extracted dataset folder.
DATA_DIR = 'dataset/static'
df_train = pq.read_table(f'{DATA_DIR}/train.parquet').to_pandas()
df_test  = pq.read_table(f'{DATA_DIR}/test.parquet').to_pandas()

feat_cols = [c for c in df_train.columns if c.startswith('feature_')]
def split_X_y(df):
    return df[feat_cols].values.astype('float32'), df['label'].values.astype('int32')
X_train, y_train = split_X_y(df_train)
X_test,  y_test  = split_X_y(df_test)
print(f'train {X_train.shape}  test {X_test.shape}')

## Train Random Forest (200 trees)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, random_state=42,
)
model.fit(X_train, y_train)

## Evaluate

In [ ]:
p_test = model.predict_proba(X_test)[:, 1]
print(f'RandomForest  test ROC AUC = {roc_auc_score(y_test, p_test):.4f}    '
      f'PR AUC = {average_precision_score(y_test, p_test):.4f}    '
      f'F1 = {f1_score(y_test, p_test > 0.5):.4f}')